# CodeGen Week 3 — clean control experiment (generation only)

Three arms on the **same 318 train items**, one model load, sequential:
`zs_rerun` → `random_k3` → `hybrid_k3`. Prompts are frozen in the input package (built on Windows
with the canonical pipeline). This notebook only **generates raw outputs**: no database, no
credentials, no scoring. Scoring happens on Windows against the read-only Atlas account.

Settings: **Accelerator = GPU T4 x2** (the runner pins itself to GPU 0), **Internet = On**, notebook **Private**.
Inputs: the private dataset holding `week3_control_v1.zip` (Kaggle unzips it), and optionally the
GGUF model as a second private dataset. To resume a stopped run, also attach this notebook's
previous version output as an input.

In [ ]:
# ---- settings (the only cell you may need to edit) ----
RUN_FULL = True            # False: smoke test only
SMOKE_N = 3                # items per arm in the smoke test (also regenerated in reverse order)
DETERMINISM_N = 5          # items per arm regenerated after the full run
LLAMA_VERSION = "0.3.36"   # must equal the version used to budget the prompts (checked again by the runner)
MODEL_FILE = "qwen2.5-coder-3b-instruct-q4_k_m.gguf"
MODEL_SHA256 = "724fb256bec1ff062b2f65e4569e871ad2e95ab2a3989723d1769c54294730b7"
HF_REPO = "Qwen/Qwen2.5-Coder-3B-Instruct-GGUF"

In [ ]:
import glob, hashlib, json, os, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
PKG = WORK / "pkg"
OUT = WORK / "week3_control_outputs"
SMOKE = WORK / "week3_control_smoke"
LOGS = WORK / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def sh(cmd, log=None, check=True, env=None):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, env=env)
    text = p.stdout + p.stderr
    if log:
        with open(LOGS / log, "a") as f:
            f.write(text)
    print(text[-4000:])
    if check and p.returncode:
        raise SystemExit(f"command failed ({p.returncode}): see logs/{log}")
    return text

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()

sh("nvidia-smi", log="nvidia_smi.txt")
assert shutil.which("nvidia-smi"), "no GPU: set Accelerator = GPU T4 in the notebook settings"

## 1. Copy the read-only input package into working storage and verify every hash

In [ ]:
found = [Path(p).parent for p in glob.glob("/kaggle/input/**/package_manifest.json", recursive=True)
         if Path(p).parent.name != "pkg"]          # skip the working copy inside an attached old output
assert len(found) == 1, f"expected exactly one input package, found {found}"
if PKG.exists():
    shutil.rmtree(PKG)
shutil.copytree(found[0], PKG)
for line in (PKG / "SHA256SUMS.txt").read_text().splitlines():
    h, name = line.split("  ", 1)
    assert sha256(PKG / name) == h, f"hash mismatch: {name}"
man = json.loads((PKG / "package_manifest.json").read_text())
print(man["package"], man["n_items"], "items; arms", man["arm_order"], "; random seed", man["random_seed"])
print("all package hashes verified")
assert not glob.glob("/kaggle/input/**/.env", recursive=True), "an .env file is attached as input: remove it"

## 2. Model weights (attached dataset, else Hugging Face download); SHA-256 must match

In [ ]:
cands = glob.glob(f"/kaggle/input/**/{MODEL_FILE}", recursive=True)
if cands:
    MODEL = cands[0]
else:
    from huggingface_hub import hf_hub_download
    MODEL = hf_hub_download(HF_REPO, MODEL_FILE, local_dir="/tmp/model")   # /tmp: not part of the output
digest = sha256(MODEL)
print(MODEL, digest)
assert digest == MODEL_SHA256, "model sha256 differs from the one used in every previous run; stopping"

## 3. CUDA build of llama-cpp-python (pinned); reuses a wheel from an attached input when present

In [ ]:
wheels = glob.glob(f"/kaggle/input/**/llama_cpp_python-{LLAMA_VERSION}-*.whl", recursive=True)
WHEELS = WORK / "wheels"
WHEELS.mkdir(exist_ok=True)
if not wheels:
    cc = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.split()[0].replace(".", "")
    env = dict(os.environ, CMAKE_ARGS=f"-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES={cc}", FORCE_CMAKE="1")
    if os.path.exists("/usr/local/cuda/bin/nvcc"):
        env["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
        env["PATH"] = "/usr/local/cuda/bin:" + env["PATH"]
    sh("nvcc --version || true", log="llama_build.log", env=env)
    t0 = time.time()
    sh([sys.executable, "-m", "pip", "wheel", f"llama-cpp-python=={LLAMA_VERSION}", "--no-deps",
        "--no-cache-dir", "-w", str(WHEELS)], log="llama_build.log", env=env)
    print(f"built in {time.time() - t0:.0f}s")
    wheels = glob.glob(str(WHEELS / f"llama_cpp_python-{LLAMA_VERSION}-*.whl"))
else:
    shutil.copy(wheels[0], WHEELS)
sh([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", wheels[0]], log="pip_install.log")
sh([sys.executable, "-m", "pip", "install", "diskcache>=5.6.1", "jinja2>=2.11.3", "numpy>=1.20.0",
    "typing-extensions>=4.5.0"], log="pip_install.log")
sh([sys.executable, "-c", "import llama_cpp; print(llama_cpp.__version__, "
    "'gpu_offload', llama_cpp.llama_supports_gpu_offload())"], log="pip_install.log")
sh([sys.executable, "-m", "pip", "freeze"], log="pip_freeze.txt")

## 4. Smoke test (separate directory; never mixed into the real run)

In [ ]:
def run_generator(out_dir, extra, log):
    cmd = [sys.executable, str(PKG / "kaggle_generate.py"), "--package", str(PKG), "--out", str(out_dir),
           "--model", MODEL, "--n-gpu-layers", "-1", "--require-gpu"] + extra
    # one GPU only: with two visible T4s llama.cpp would split the layers across both cards
    return sh(cmd, log=log, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))

if SMOKE.exists():
    shutil.rmtree(SMOKE)
text = run_generator(SMOKE, ["--limit", str(SMOKE_N), "--determinism-check", str(SMOKE_N)], "smoke.log")
offload = re.findall(r"offloaded (\d+)/(\d+) layers to GPU", text)
print("layer offload lines:", offload)
# the hard gate is the runner's --require-gpu check (GPU memory must grow >= 1.5 GB at load);
# the llama.cpp log line, when present, must show every layer offloaded
assert all(a == b for a, b in offload), "not every layer is on the GPU"
for arm in man["arm_order"]:
    recs = [json.loads(l) for l in open(SMOKE / arm / "generations.jsonl")]
    assert len(recs) == SMOKE_N and len({r["id"] for r in recs}) == SMOKE_N
    for r in recs:
        t = (r["raw_text"] or "").strip().strip("`").removeprefix("json").strip()
        try:
            json.loads(t)
            ok = "parses"
        except Exception:
            ok = "does NOT parse as plain JSON (the Windows scorer decides)"
        print(arm, r["id"], r["output_tokens"], "tokens", f"{r['generation_wall_s']}s", ok)
print(json.load(open(SMOKE / "determinism_check.json")))

## 5. Full run: all 318 items per arm, sequential, checkpointed per item (resumes automatically)

In [ ]:
prev = [Path(p).parent for p in glob.glob("/kaggle/input/**/week3_control_outputs/run_manifest.json", recursive=True)]
if prev and not OUT.exists():
    shutil.copytree(prev[0], OUT)          # resume from the attached previous version's output
    print("resuming from", prev[0])
if RUN_FULL:
    run_generator(OUT, ["--determinism-check", str(DETERMINISM_N)], "full_run.log")

## 6. Completeness check and downloadable ZIP (outputs, manifests, hashes, logs)

In [ ]:
status = {}
for arm in man["arm_order"]:
    p = OUT / arm / "generations.jsonl"
    recs = [json.loads(l) for l in open(p)] if p.exists() else []
    ids = [r["id"] for r in recs]
    assert len(ids) == len(set(ids)), f"{arm}: duplicate ids"
    status[arm] = {"generated": len(ids), "of": man["n_items"], "complete": set(ids) == set(man["item_ids"]),
                   "provider_error_records": sum(1 for _ in open(OUT / arm / "provider_errors.jsonl"))
                   if (OUT / arm / "provider_errors.jsonl").exists() else 0}
print(json.dumps(status, indent=1))
for d in (OUT, SMOKE):
    if d.exists():
        shutil.copytree(LOGS, d / "notebook_logs", dirs_exist_ok=True)
        (d / "completeness.json").write_text(json.dumps(status, indent=1))
        files = sorted(p for p in d.rglob("*") if p.is_file() and p.name != "SHA256SUMS.txt")
        (d / "SHA256SUMS.txt").write_text("".join(f"{sha256(p)}  {p.relative_to(d).as_posix()}\n" for p in files))
ZIP = WORK / "week3_control_outputs.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for d in (OUT, SMOKE):
        if d.exists():
            for p in sorted(d.rglob("*")):
                if p.is_file():
                    z.write(p, p.relative_to(WORK).as_posix())
print(ZIP, ZIP.stat().st_size, "bytes; sha256", sha256(ZIP))